In [67]:
# Documentation: This code is for preprocessing the text and turning it into json
# Step 1 - Read text file
# Step 2 - Clean text using regex
# Step 3 - Parse structured text(section detection)
# Step 4 - Save dictionary to JSON

import re #For regex
import json #For converting dictionary to JSON file
import unicodedata #For fixing strange characters and symbols

# Step 1 - Read text file
def read_text_file(file_path):#Creates a function, expects parameter file_path
    with open(file_path, "r", encoding = "utf-8")as file:#Open file_path, in read mode, in that encoding
        return file.read()#Reads the entire file as 1 big string


#Step 2 - Clean Text using regex
def clean_text(text): #This function receives the full document string
    #Normalize unicode(fix weird characters)
    text = unicodedata.normalize("NFKD", text)

    #Remove website header
    text = re.sub(r"www\.lawcommission\.gov\.np", "", text)

    #Remove page numbers
    text = re.sub(r"\n\s*\d+\s*\n", "\n", text)#\n=new line, \s*=spaces, \d+=1 or more digits

    #Remove multiple spaces(fix Weird Spacing)
    text = re.sub(r"[ \t]+", " ", text)#Removes multiple spaces, tabs keeps new lines
    
    return text.strip()#Removes leading spaces, trailing spaces
    

In [68]:
# Step 3 - Parse structured text(section detection)
def parse_structure(text):#Takes cleaned document string
    lines = text.split("\n") #Breaks big string into a list eg. Chapter-1. ..., 1. ..., (a)...

    #Initializing Variables
    data = {} #Final Dictionary that will become JSON
    current_chapter = None #Remembers which chapter we're in
    current_section = None #Remembers which section we're in

    for line in lines:
        line = line.strip()#Loops line by line, removes extra space from start/end

        # Detect Chapter
        chapter_match = re.match(r"CHAPTER-\d+", line)#Checks if line starts with Chapter-1/Chapter-2/...
        #Here, CHAPTER = literal text and \d+ = 1 or more digits 
        if chapter_match:
            current_chapter = line
            data[current_chapter] = {}
            continue
        #Here, The chapter name is saved, a new dictionary is created inside data and we skip to the next line
        #eg. { "Chapter-1":{} }
        
        # Detect Section (e.g., 1. , 2. , 3.)
        section_match = re.match(r"^(\d+)\.\s+([A-Z].*)", line) #^(\d+)\. means number + dot at start of line,\s+ → at least one space after dot([A-Z].*) → content starts with an uppercase letter 
        if section_match:
            section_number = section_match.group(1)
            if int(section_number) > 1000: #Skip numbers like 2031, 2034
                continue
            section_title = section_match.group(2)

            current_section = f"Section {section_number}"#Section Key
            
            if current_chapter: #Store Section as dictionary
                data[current_chapter][current_section] = {
                    "title": section_title,
                    "content": ""
                }
            else:
                data[current_section] = {
                    "title": section_title,
                    "content": ""
                }
            continue

        # Add content under current section
        if current_section:
            if current_chapter and current_section in data.get(current_chapter, {}):
                data[current_chapter][current_section]["content"] += " " + line
            elif not current_chapter and current_section in data:
                data[current_section]["content"] += " " + line

    return data

In [69]:
# Step 4 - Save to json
def save_to_json(data, output_file): #Opens output_file for writing in UTF-8(needed for nepali characters)
    with open(output_file, "w", encoding="utf-8") as json_file:
        json.dump(data, json_file, indent=4, ensure_ascii=False)#converts dictionary to json, formats it properly

#Main
if __name__ == "__main__": #Ensures this code only runs if this script is directly executed
    #For Evidence Act
    input_file = r"C:\Users\bilak\Project\Evidence_Act.txt"    
    output_file = "Evidence_Act.json"

    #For Constitution
    input_file_constitution = r"C:\Users\bilak\Project\Constitution.txt"    
    output_file_constitution = "Constitution.json"

    #For Criminal Acts
    #Penal Act
    input_file_criminal1 = r"C:\Users\bilak\Project\CriminalCode1_Penal_Code_Act.txt"
    output_file_criminal1 = "Criminal1.json"

    #Sentencing and Execution Act
    input_file_criminal2 = r"C:\Users\bilak\Project\CriminalCode2_Sentencing_and_Execution_Act.txt"
    output_file_criminal2 = "Criminal2.json"

    #Criminal Procedure Act
    input_file_criminal3 = r"C:\Users\bilak\Project\CriminalCode3_Criminal_Procedure_Act.txt"
    output_file_criminal3 = "Criminal3.json"

    #Civil Code and Procedure
    input_file_civil = r"C:\Users\bilak\Project\Civil_Code_and_Procedure.txt"
    output_file_civil = "Civil.json"
    
    #if u wanna add more text files, just add 1 more input and output per file and keep it in the same folder

    #For Processing Evidence Act
    raw_text = read_text_file(input_file)
    cleaned_text = clean_text(raw_text)
    structured_data = parse_structure(cleaned_text)
    save_to_json(structured_data, output_file)
    print("Evidence Act JSON file created successfully.")

    #For Processing Constitution
    raw_text = read_text_file(input_file_constitution)
    cleaned_text = clean_text(raw_text)
    structured_data = parse_structure(cleaned_text)
    save_to_json(structured_data, output_file_constitution)
    print("Constitution JSON file created successfully.")

    #For Processing Criminal Acts
    #Penal Act
    raw_text = read_text_file(input_file_criminal1)
    cleaned_text = clean_text(raw_text)
    structured_data = parse_structure(cleaned_text)
    save_to_json(structured_data, output_file_criminal1)

    #Sentencing and Execution Act
    raw_text = read_text_file(input_file_criminal2)
    cleaned_text = clean_text(raw_text)
    structured_data = parse_structure(cleaned_text)
    save_to_json(structured_data, output_file_criminal2)

    #Criminal Procedure Act
    raw_text = read_text_file(input_file_criminal3)
    cleaned_text = clean_text(raw_text)
    structured_data = parse_structure(cleaned_text)
    save_to_json(structured_data, output_file_criminal3)
    print("Criminal Act JSON file generated successfully.")

    #For Civil Code and Procedure
    raw_text = read_text_file(input_file_civil)
    cleaned_text = clean_text(raw_text)
    structured_data = parse_structure(cleaned_text)
    save_to_json(structured_data, output_file_civil)
    print("Civil Code and Procedure JSON file generated successfully.")

Evidence Act JSON file created successfully.
Constitution JSON file created successfully.
Criminal Act JSON file generated successfully.
Civil Code and Procedure JSON file generated successfully.


In [70]:
#Deliverables:
#Takes text, Cleans it and converts to structured JSON
#Chapters, sections, and content all properly nested
#Creates a solid foundation for semantic search, retrieval, and eventually LLM integration, which is what you can do next. Maybe try out another law too.